# Email Sending Agent using LangChain Tools
### Real-world tool calling — give a natural language instruction, the agent figures out who to email, what to write, and sends it.

**Stack:** Google Gemini 2.5 Flash + Gmail SMTP + LangChain Tools

---

## Step 1 — Install Dependencies

In [11]:
!pip install -q langchain langchain-google-genai langchain-core python-dotenv


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2 — Setup API Key

Set your Gemini API key and Gmail credentials below.

> **IMPORTANT:** For Gmail, you need an **App Password**, NOT your real Gmail password.
> 1. Go to [myaccount.google.com](https://myaccount.google.com)
> 2. Security → 2-Step Verification (enable it first)
> 3. Search **App passwords** → create one for Mail
> 4. Paste the 16-character password below

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
os.environ['SENDER_EMAIL'] = os.getenv("SENDER_EMAIL")
os.environ['SENDER_APP_PASSWORD'] = os.getenv("SENDER_APP_PASSWORD")

## Step 3 — Import Everything

In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import json, os

## Step 4 — Define the Tools

We define three tools. The agent decides which to call and in what order based on the descriptions.

- `compose_email` — drafts and previews the email before sending
- `send_email` — actually sends it via Gmail SMTP
- `check_email_status` — confirms delivery after sending

In [14]:
@tool
def compose_email(recipient_email: str, subject: str, body: str) -> str:
    """
    Compose an email given a recipient email address, a subject line, and a body message.
    Returns a confirmation string with the composed email details for review before sending.
    """
    composed = {"to": recipient_email, "subject": subject, "body": body}
    return json.dumps(composed)


@tool
def send_email(recipient_email: str, subject: str, body: str) -> str:
    """
    Send an email to a given recipient email address with a given subject and body.
    Uses Gmail SMTP to deliver the email. Returns success or error message.
    """
    sender_email = os.environ['SENDER_EMAIL']
    app_password = os.environ['SENDER_APP_PASSWORD']
    try:
        msg = MIMEMultipart()
        msg['From'] = sender_email
        msg['To'] = recipient_email
        msg['Subject'] = subject
        msg.attach(MIMEText(body, 'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(sender_email, app_password)
            server.sendmail(sender_email, recipient_email, msg.as_string())
        return f'Email successfully sent to {recipient_email} with subject: {subject}'
    except smtplib.SMTPAuthenticationError:
        return 'Authentication failed. Make sure you are using a Gmail App Password, not your regular password.'
    except smtplib.SMTPRecipientsRefused:
        return f'Recipient {recipient_email} was refused. Check if the email address is correct.'
    except Exception as e:
        return f'Failed to send email: {str(e)}'


@tool
def check_email_status(recipient_email: str) -> str:
    """
    Check and confirm the status of an email that was sent to a recipient email address.
    Returns a confirmation message.
    """
    return f'Confirmed: An email was dispatched to {recipient_email}. It should arrive within a few minutes.'


print('Tools defined:')
print(f'  {compose_email.name}: {compose_email.description[:70]}...')
print(f'  {send_email.name}: {send_email.description[:70]}...')
print(f'  {check_email_status.name}: {check_email_status.description[:70]}...')

Tools defined:
  compose_email: Compose an email given a recipient email address, a subject line, and ...
  send_email: Send an email to a given recipient email address with a given subject ...
  check_email_status: Check and confirm the status of an email that was sent to a recipient ...


## Step 5 — Initialise the Model and Bind Tools

`bind_tools()` makes the model aware of what tools exist. The model reads each tool's `name` and `description` to decide when and how to use them.

In [15]:
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.environ['GEMINI_API_KEY']
)

tools = [compose_email, send_email, check_email_status]
tool_map = {t.name: t for t in tools}  # dict for easy lookup by name

llm_with_tools = llm.bind_tools(tools)

print('Model ready with tools bound:', [t.name for t in tools])

Model ready with tools bound: ['compose_email', 'send_email', 'check_email_status']


## Step 6 — The Agent Loop

This is the heart of the project. The loop:
1. Sends the user instruction to the model
2. Model replies with tool call requests (not text)
3. We run each requested tool
4. Append the results back to messages
5. Send everything back to the model
6. Repeat until the model gives a final plain text answer (no more tool calls)

This `while True` loop **is** the agent.

In [16]:
def run_agent(user_instruction: str):
    print(f'User: {user_instruction.strip()}')
    print('-' * 60)

    messages = [HumanMessage(content=user_instruction)]
    step = 1

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # No tool calls = model has a final answer
        if not response.tool_calls:
            print(f'\nAgent Final Answer:')
            print(response.content)
            break

        # Model wants to call tools
        print(f'\nStep {step} — Model requested {len(response.tool_calls)} tool call(s):')

        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            print(f'  Calling: [{tool_name}]')
            print(f'  Args:    {tool_args}')

            tool_fn = tool_map[tool_name]
            tool_result = tool_fn.invoke(tool_call)
            messages.append(tool_result)
            print(f'  Result:  {tool_result.content}')

        step += 1

    return messages

## Step 7 — Run It!

Change the email address and instruction below to whatever you want.
The agent will figure out the subject, write the body, and send it.

In [17]:
instruction = """
Send a friendly email to sumukh.m2006@gmail.com
telling them that I have just completed my LangChain course
and I am excited to start building AI projects.
Keep the tone casual and enthusiastic. Sign it as a fellow developer.
"""

messages = run_agent(instruction)

User: Send a friendly email to sumukh.m2006@gmail.com
telling them that I have just completed my LangChain course
and I am excited to start building AI projects.
Keep the tone casual and enthusiastic. Sign it as a fellow developer.
------------------------------------------------------------

Step 1 — Model requested 1 tool call(s):
  Calling: [send_email]
  Args:    {'subject': 'Excited to build AI projects!', 'body': "Hey Sumukh,\n\nI just finished my LangChain course and I'm super excited to start building AI projects! Let's connect soon and share ideas.\n\nBest,\nA fellow developer", 'recipient_email': 'sumukh.m2006@gmail.com'}
  Result:  Email successfully sent to sumukh.m2006@gmail.com with subject: Excited to build AI projects!

Agent Final Answer:
I have sent a friendly email to sumukh.m2006@gmail.com with the subject "Excited to build AI projects!" and a casual, enthusiastic message about completing your LangChain course and your excitement to start building AI projects. I sig

## Step 8 — Inspect the Full Message History

See exactly what happened internally — every message, tool call, and result. This is the messages list you studied in the Tool Calling lecture.

In [18]:
print('\nFull Message History:')
print('=' * 60)
for i, msg in enumerate(messages):
    msg_type = type(msg).__name__
    if hasattr(msg, 'content') and msg.content:
        preview = str(msg.content)[:120]
    else:
        preview = '[no text — tool call request]'
    print(f'[{i}] {msg_type}: {preview}')
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f'      Tool: {tc["name"]} | Args: {tc["args"]}')
print('=' * 60)


Full Message History:
[0] HumanMessage: 
Send a friendly email to sumukh.m2006@gmail.com
telling them that I have just completed my LangChain course
and I am ex
[1] AIMessage: [no text — tool call request]
      Tool: send_email | Args: {'subject': 'Excited to build AI projects!', 'body': "Hey Sumukh,\n\nI just finished my LangChain course and I'm super excited to start building AI projects! Let's connect soon and share ideas.\n\nBest,\nA fellow developer", 'recipient_email': 'sumukh.m2006@gmail.com'}
[2] ToolMessage: Email successfully sent to sumukh.m2006@gmail.com with subject: Excited to build AI projects!
[3] AIMessage: I have sent a friendly email to sumukh.m2006@gmail.com with the subject "Excited to build AI projects!" and a casual, en


## Step 9 — Try More Instructions

See how the agent handles different types of email tasks.

In [19]:
# Formal follow-up email
run_agent("""
Send a formal follow-up email to sumukh.m2006@gmail.com
asking about the status of an internship application
submitted last week. Sign it as a CS student from VIT.
""")

User: Send a formal follow-up email to sumukh.m2006@gmail.com
asking about the status of an internship application
submitted last week. Sign it as a CS student from VIT.
------------------------------------------------------------

Agent Final Answer:
[{'type': 'text', 'text': 'I can help you with that. What is your name, please? I need it to sign the email and include it in the subject line.', 'extras': {'signature': 'Co0JARFNMg8lGAq2hMm1EP5TJdfNY3XZ1M2RtPAUBFUzxR8PwmkAwD+KFT4qmdSGHNoxgrwo9ucKKaBv8UEHHlBkKxWi4a6dk9sZMSzWQ7t1HjUVpAkmMTIaGFnZPQxqUnwGO+aGuUpfpbRFaGCo9LZqJ745vS0lqWucAe2hZ9jvvVB1k8B72gTko3yRi1d5+7U326gL3pt8Wm7eULT5m+jqdex34cbSBqdQQBlDW1vxRN9zGCnHCB5yrYNSnLjmS1lNBDRFvjsmLYJnBW2RBby/ePmpXQcDH6KbnrQTK2KAds0C6jV8Lph6W7cZG7F+yC0lRIzgKfB3eZ4vMPbL3jHxEZHSQOiGylx8nhNXWhI3R3U8SnglPgH3AyAxlr94eHIHcfUOSt6vuo9j1F6FVbCWUsJnr3D7aRpEfDK6zL27Ut6gCEzjulPmaCxHCVcX+/B1Fv2jk4Hln/lnTFX6yRaMkL5LDfyEvS5iot2ic5+h/ZIGcnkeF5ztgyJLz40JBq8ZRGuA0xzL/5nPvl0KyYCJvMWne+4TzFkL9PayfpBf2c/z5DptqEqvUwQ+bKYmm

[HumanMessage(content='\nSend a formal follow-up email to sumukh.m2006@gmail.com\nasking about the status of an internship application\nsubmitted last week. Sign it as a CS student from VIT.\n', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'type': 'text', 'text': 'I can help you with that. What is your name, please? I need it to sign the email and include it in the subject line.', 'extras': {'signature': 'Co0JARFNMg8lGAq2hMm1EP5TJdfNY3XZ1M2RtPAUBFUzxR8PwmkAwD+KFT4qmdSGHNoxgrwo9ucKKaBv8UEHHlBkKxWi4a6dk9sZMSzWQ7t1HjUVpAkmMTIaGFnZPQxqUnwGO+aGuUpfpbRFaGCo9LZqJ745vS0lqWucAe2hZ9jvvVB1k8B72gTko3yRi1d5+7U326gL3pt8Wm7eULT5m+jqdex34cbSBqdQQBlDW1vxRN9zGCnHCB5yrYNSnLjmS1lNBDRFvjsmLYJnBW2RBby/ePmpXQcDH6KbnrQTK2KAds0C6jV8Lph6W7cZG7F+yC0lRIzgKfB3eZ4vMPbL3jHxEZHSQOiGylx8nhNXWhI3R3U8SnglPgH3AyAxlr94eHIHcfUOSt6vuo9j1F6FVbCWUsJnr3D7aRpEfDK6zL27Ut6gCEzjulPmaCxHCVcX+/B1Fv2jk4Hln/lnTFX6yRaMkL5LDfyEvS5iot2ic5+h/ZIGcnkeF5ztgyJLz40JBq8ZRGuA0xzL/5nPvl0KyYCJvMWne+4TzFkL9PayfpBf2c/z5DptqEqvUw

In [20]:
# Thank you email
run_agent("""
Send a short thank you email to sumukh.m2006@gmail.com
thanking a professor for writing a recommendation letter.
Keep it warm, genuine, and under 5 sentences.
""")

User: Send a short thank you email to sumukh.m2006@gmail.com
thanking a professor for writing a recommendation letter.
Keep it warm, genuine, and under 5 sentences.
------------------------------------------------------------



Step 1 — Model requested 1 tool call(s):
  Calling: [send_email]
  Args:    {'recipient_email': 'sumukh.m2006@gmail.com', 'body': 'Dear Professor, I hope this email finds you well. I wanted to express my sincere gratitude for writing a recommendation letter for me. Your support means a lot, and I truly appreciate you taking the time to do so. I will keep you updated on the outcome. Thank you once again!', 'subject': 'Thank You for Your Recommendation!'}
  Result:  Email successfully sent to sumukh.m2006@gmail.com with subject: Thank You for Your Recommendation!

Agent Final Answer:
Your thank you email has been sent to sumukh.m2006@gmail.com!


[HumanMessage(content='\nSend a short thank you email to sumukh.m2006@gmail.com\nthanking a professor for writing a recommendation letter.\nKeep it warm, genuine, and under 5 sentences.\n', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_email', 'arguments': '{"recipient_email": "sumukh.m2006@gmail.com", "body": "Dear Professor, I hope this email finds you well. I wanted to express my sincere gratitude for writing a recommendation letter for me. Your support means a lot, and I truly appreciate you taking the time to do so. I will keep you updated on the outcome. Thank you once again!", "subject": "Thank You for Your Recommendation!"}'}, '__gemini_function_call_thought_signatures__': {'aa4494b9-f498-4467-9064-6b7cc959ef95': 'CrQDARFNMg9trS1Tbg2aFRhEpPZuiaSLJUin2fCQYld6Vx8DWlvdLnxx9VpcxG//BYuO1v7d1M/HHZCPrBtfi6Ip2deZgHcLJyMmh9UTFcMRsc5aJdjxb7fxedQ0ZD0t67ukG/sC/zIsF1ngie0HIOg/Kfa2zDVVJgy4eCO5j6Mw5uDp/2awcQzpLd5asMuBdhj

---
## How This Connects to What You Studied

| Concept | Where it appears |
|---|---|
| `@tool` decorator | `compose_email`, `send_email`, `check_email_status` |
| `tool.name` / `.description` / `.args` | Printed in Step 4 |
| `bind_tools([...])` | Step 5 — model becomes aware of tools |
| `response.tool_calls` | Checked in the while loop |
| `tool_fn.invoke(tool_call)` | Runs the actual tool with model-provided args |
| `ToolMessage` | Appended back to messages after each tool runs |
| Agent loop pattern | The `while True` in `run_agent()` IS the agent |

> The key insight: the model never sends the email itself. It only says *call send_email with these args*. You run it, give the result back, and the model confirms it's done. That back-and-forth is tool calling.